# 비즈니스 문제 정의
현재 상황
- 다이캐스팅 공정은 고압·고속으로 용융 금속을 주입하여 정밀 부품을 생산하는 핵심 제조 공정

- 불량 판정이 여전히 육안 검사에 의존

- 불량 발생 후 원인 추적이 어려워 사후 대응 중심의 관리 체계 운영

- 공정 데이터는 축적되고 있으나, 품질 데이터와 효과적으로 연계되지 않음

- 불량 발생 시 재작업·스크랩 비용 증가 및 납기 지연 발생

## 비즈니스 이슈

- 불량 유형별 발생 원인에 대한 정량적 분석 부족

- 실시간 불량 예측 시스템 부재

- 공정 변수와 품질 결과 간 관계 파악 미흡

- Shot 단위 품질 추적 체계 부족

- 불량 발생 후 대응 중심 → 예방 중심 관리 필요

## 해결 필요성

- 머신러닝 기반 불량 예측 시스템 도입

- 주요 공정 변수의 영향력 분석

- 실시간 조기 경보 시스템 구축

- 불량률 감소 → 생산성 향상

- 데이터 기반 공정 최적화 체계 수립

### 주요 질문

"어떤 공정 조건에서 불량이 발생할 가능성이 높은가?"



In [32]:
# 데이터 처리 및 분석
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
import platform

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

In [33]:
df = pd.read_csv("C:/Users/hi/OneDrive/바탕 화면/data/DieCasting_Quality_Raw_Data.csv")
df

,Process,Process.1,Process.2,Process.3,Process.4,Process.5,Process.6,Process.7,Process.8,Process.9,Process.10,Process.11,Process.12,Process.13,Process.14,Process.15,Process.16,Sensor,Sensor.1,Sensor.2,Sensor.3,Sensor.4,Sensor.5,Sensor.6,Sensor.7,Sensor.8,Sensor.9,Sensor.10,Sensor.11,Sensor.12,Sensor.13,Defects,Defects.1,Defects.2,Defects.3,Defects.4,Defects.5,Defects.6,Defects.7,Defects.8,Defects.9,Defects.10,Defects.11,Defects.12,Defects.13,Defects.14,Defects.15,Defects.16,Defects.17,Defects.18,Defects.19,Defects.20,Defects.21,Defects.22,Defects.23,Defects.24,Defects.25
0,id,Product_Type,Shot,Velocity_1,Velocity_2,Velocity_3,High_Velocity,Cylinder_Pressure,Rapid_Rise_Time,Biscuit_Thickness,Clamping_Force,Cycle_Time,Pressure_Rise_Time,Casting_Pressure,Spray_Time,Spray_1_Time,Spray_2_Time,Melting_Furnace_Temp,Air_Pressure,Air_Pressure_Min,Air_Pressure_Max,Coolant_Temp,Coolant_Temp_Min,Coolant_Temp_Max,Coolant_Pressure,Factory_Temp,Factory_Temp_Min,Factory_Temp_Max,Factory_Humidity,Factory_Humidity_Min,Factory_Humidity_Max,Short_Shot_1,Bubble_1,Exfoliation_1,Blow_Hole_1,Stain_1,Dent_1,Deformation_1,Contamination_1,Impurity_1,Crack_1,Scratch_1,Buring_Mark_1,Inclusions_1,Short_Shot_2,Bubble_2,Exfoliation_2,Blow_Hole_2,Stain_2,Dent_2,Deformation_2,Contamination_2,Impurity_2,Crack_2,Scratch_2,Buring_Mark_2,Inclusions_2
1,1,1,1,0.144,0.17,0.188,2.134,214,0.008,10,258,20.7,0.044,1037,7.8,0.7,0.8,695.0,6.3,3,9,26.0,10,50,2.71,32.9,18.0,22.0,58.4,18.0,22.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1002,1,2,0.144,0.17,0.182,2.124,217,0.008,11,257,20.7,0.044,1052,7.8,0.7,0.8,696.4,6.3,3,9,26.1,10,50,2.69,32.9,18.0,22.0,58.2,18.0,22.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,2003,1,3,0.144,0.17,0.182,2.116,214,0.008,11,257,20.8,0.041,1037,7.8,0.7,0.8,696.4,6.3,3,9,26.1,10,50,2.69,32.9,18.0,22.0,58.2,18.0,22.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,3004,1,4,0.144,0.17,0.182,2.137,217,0.008,11,257,20.7,0.043,1051,7.8,0.7,0.8,696.4,6.3,3,9,26.1,10,50,2.69,32.9,18.0,22.0,58.2,18.0,22.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7531,7530659,2,659,0.15,0.166,0.21,2.492,265,0.011,17,381,36.2,0.033,595,12.1,2.0,2.0,667.4,6.7,3,9,28.1,10,50,2.62,32.3,18.0,22.0,69.4,18.0,22.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7532,7531660,2,660,0.144,0.174,0.206,2.514,264,0.011,16,381,36.2,0.041,595,12.1,2.0,2.0,667.8,7.0,3,9,28.1,10,50,2.62,32.2,18.0,22.0,69.5,18.0,22.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7533,7532660,2,660,0.144,0.174,0.206,2.514,264,0.011,16,381,36.2,0.041,595,12.1,2.0,2.0,667.8,7.0,3,9,28.1,10,50,2.62,32.2,18.0,22.0,69.5,18.0,22.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7534,7533661,2,661,0.147,0.174,0.204,2.532,265,0.012,18,382,36.2,0.036,596,12.1,2.0,2.0,668.5,6.8,3,9,28.1,10,50,2.62,32.2,18.0,22.0,69.6,18.0,22.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [34]:
df.shape

(7536, 57)

In [35]:
# Defect 변수
defect_cols = df.columns[-26:]

# Sensor 변수
sensor_cols = [col for col in df.columns if "Sensor" in str(col)]

# Process 변수
process_cols = [col for col in df.columns 
                if col not in defect_cols 
                and col not in sensor_cols]

print("Process 변수 개수 :", len(process_cols))
print("Sensor 변수 개수  :", len(sensor_cols))
print("Defect 변수 개수  :", len(defect_cols))

Process 변수 개수 : 17
Sensor 변수 개수  : 14
Defect 변수 개수  : 26


In [36]:
df.info()
# 모든 칼럼이 object타입

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7536 entries, 0 to 7535
Data columns (total 57 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Process     7536 non-null   object
 1   Process.1   7536 non-null   object
 2   Process.2   7536 non-null   object
 3   Process.3   7536 non-null   object
 4   Process.4   7536 non-null   object
 5   Process.5   7536 non-null   object
 6   Process.6   7536 non-null   object
 7   Process.7   7536 non-null   object
 8   Process.8   7536 non-null   object
 9   Process.9   7536 non-null   object
 10  Process.10  7536 non-null   object
 11  Process.11  7536 non-null   object
 12  Process.12  7536 non-null   object
 13  Process.13  7536 non-null   object
 14  Process.14  7536 non-null   object
 15  Process.15  7536 non-null   object
 16  Process.16  7536 non-null   object
 17  Sensor      7536 non-null   object
 18  Sensor.1    7536 non-null   object
 19  Sensor.2    7536 non-null   object
 20  Sensor.3

In [37]:
df.describe()

,Process,Process.1,Process.2,Process.3,Process.4,Process.5,Process.6,Process.7,Process.8,Process.9,Process.10,Process.11,Process.12,Process.13,Process.14,Process.15,Process.16,Sensor,Sensor.1,Sensor.2,Sensor.3,Sensor.4,Sensor.5,Sensor.6,Sensor.7,Sensor.8,Sensor.9,Sensor.10,Sensor.11,Sensor.12,Sensor.13,Defects,Defects.1,Defects.2,Defects.3,Defects.4,Defects.5,Defects.6,Defects.7,Defects.8,Defects.9,Defects.10,Defects.11,Defects.12,Defects.13,Defects.14,Defects.15,Defects.16,Defects.17,Defects.18,Defects.19,Defects.20,Defects.21,Defects.22,Defects.23,Defects.24,Defects.25
count,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7446,7446,7446,7446,7446,7446,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536,7536
unique,7536,3,1272,34,28,46,320,23,19,25,43,62,17,66,33,8,9,738,27,2,2,24,2,2,23,68,2,2,218,2,2,4,3,4,5,4,3,4,3,3,3,3,3,2,4,4,4,4,2,3,4,3,3,3,2,2,3
top,id,1,44,0.144,0.168,0.182,2.524,265,0.008,11,257,36.1,0.044,596,12.1,2.0,0.8,687.0,6.2,3,9,26.5,10,50,2.74,32.0,18.0,22.0,62.0,18.0,22.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
freq,1,4207,12,1546,2540,577,204,2639,2704,1671,1004,848,1322,1492,1418,3192,3349,49,870,7535,7535,801,7535,7535,1003,1358,7445,7445,162,7445,7445,7038,7463,7362,7314,7346,7528,7427,7531,7533,7534,7533,7530,7535,7355,7528,7404,7380,7535,7531,7471,7527,7530,7533,7535,7535,7534
